In [23]:
import torch
import torch.nn as nn
import math

In [24]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_len=5000):
    super().__init__()
    pe = torch.zeros(max_seq_len, d_model)
    positions = torch.arange(0,max_seq_len).unsqueeze(1).float()
    for pos in range(max_seq_len):
      for i in range(d_model // 2):
        angle = pos / (10000 ** (2 * i / d_model))
        pe[pos, 2*i]     = math.sin(angle)
        pe[pos, 2*i + 1] = math.cos(angle)
    self.register_buffer('pe', pe)
    # divisor_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
  def forward(self, x):
    seq_len = x.size(1)
    return x + self.pe[:seq_len, :].unsqueeze(0)

# Pre LN (Modern Transformer)
class TransformerEncoder(nn.Module):
  def __init__(self, vocab_size, d_model, num_heads):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size, d_model)
    self.pe = PositionalEncoding(d_model)
    self.layerNorm1 = nn.LayerNorm(d_model, eps=1e-6)
    self.multiheadAttention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
    self.layerNorm2 = nn.LayerNorm(d_model, eps=10e-6)
    self.fnn = nn.Sequential(
      nn.Linear(d_model, 4* d_model),
      nn.ReLU(),
      nn.Linear(4*d_model, d_model)
    )

  def forward(self, x):
    # Step 1: Input Embeddings
    x = self.embeddings(x)
    x = self.pe(x)

    # Step 2: LayerNorm
    norm_x = self.layerNorm1(x)

    #Step 3: Multihead Attention
    atten_output, _ = self.multiheadAttention(query=norm_x, key=norm_x, value=norm_x)

    # Step 4: Add (Residual)
    x = x + atten_output

    # Step 5: LayerNorm
    norm_xx = self.layerNorm2(x)

    # Step 6: Feed Forward Network
    ffn_output = self.fnn(norm_xx)

    # Step 7: Add(Residual)
    x = x + ffn_output
    return x


In [25]:
# Dummy configuration
V_SIZE = 5000   # Vocabulary size
D_MOD  = 512    # Embedding dimension
HEADS  = 8      # Attention heads

model = TransformerEncoder(vocab_size=V_SIZE, d_model=D_MOD, num_heads=HEADS)

# Dummy input: Batch size, Sequence length
dummy_input = torch.randint(0, V_SIZE, (5, 55))
print("Input Shape :", dummy_input.shape)

# Forward pass
final_output = model(dummy_input)
print("Output Shape:", final_output.shape)

Input Shape : torch.Size([5, 55])
Output Shape: torch.Size([5, 55, 512])
